# 04 — Food & Recipe Recommenders: Training & Evaluation

Two models sit behind `src/diet/recommender.py`, one per section of the diet
recommendation: a **food** model (single foods, per 100 g — 672 foods) and a
**recipe** model (household recipes, per serving — 250 recipes, using each
dish's food-group make-up and its 1–10 household frequency score,
10 = the most commonly cooked dish). Both are trained by
`scripts/train_recommenders.py` (see it for the full methodology), each with a
five-way comparison — hand-tuned formula vs. a fitted `LinearRegression` vs.
an untuned `GradientBoostingRegressor` vs. a tuned `RandomForestRegressor` vs.
a tuned `HistGradientBoostingRegressor` — evaluated by `GroupKFold` held out
**by patient archetype**.

There is no real preference/interaction data in this repo — labels come from
a synthetic patient-archetype grid built on the app's own cited reference
data (`data/reference/*.yaml`), scored by a richer suitability function than
the deterministic formula it augments (saturation caps, a heme-iron
interaction, a continuous sodium penalty, and for recipes the household
familiarity weight — see `compute_food_label()` / `compute_recipe_label()`).

Run the training script first; this notebook only reads its outputs and draws
the thesis figures. Requires `pip install matplotlib pandas` — analysis-only
dependencies, not part of the runtime app, same as `03_results_figures.ipynb`.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
FIGURES = PROJECT_ROOT / "thesis" / "figures"
RESULTS = PROJECT_ROOT / "thesis" / "results"
MODELS = PROJECT_ROOT / "data" / "models"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
KINDS = ("food", "recipe")

## Load the saved evaluation

`scripts/train_recommenders.py` writes `data/models/<kind>_recommender_metrics.json`
(five-way comparison, tuned hyperparameters, permutation importance) and
`thesis/results/<kind>_recommender_comparison.csv`.

In [ ]:
metrics = {}
for kind in KINDS:
    path = MODELS / f"{kind}_recommender_metrics.json"
    assert path.exists(), "run scripts/train_recommenders.py first"
    metrics[kind] = json.loads(path.read_text(encoding="utf-8"))
    m = metrics[kind]
    print(f"{kind}: {m['n_items']} items, {m['n_rows']} rows, {m['n_features']} features, "
          f"{m['n_archetypes']} archetypes, winner = {m['winner']}")

## Metrics tables

The key ablation evidence: does the ensemble beat both the untrained
hand-tuned formula *and* a plain fitted linear model on patient archetypes
it never trained on?

In [ ]:
import pandas as pd

tables = {}
for kind in KINDS:
    rows = [
        {"model": name,
         "spearman_mean": m["spearman"]["mean"], "spearman_std": m["spearman"]["std"],
         "r2_mean": m["r2"]["mean"], "mae_mean": m["mae"]["mean"]}
        for name, m in metrics[kind]["metrics"].items()
    ]
    tables[kind] = pd.DataFrame(rows).sort_values("spearman_mean", ascending=False)
    tables[kind].to_csv(RESULTS / f"{kind}_recommender_comparison.csv", index=False)
tables["food"]

In [ ]:
tables["recipe"]

## Feature importance → thesis/figures

Permutation importance of the shipped models (drop in Spearman correlation
when one feature is shuffled), computed by the training script — model
agnostic, so it works whichever estimator wins (`HistGradientBoostingRegressor`
has no `feature_importances_`).

In [ ]:
import matplotlib.pyplot as plt

for kind in KINDS:
    importance = pd.DataFrame(metrics[kind]["feature_importance"]).set_index("feature")
    top = importance.sort_values("mean").tail(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    top["mean"].plot.barh(ax=ax, xerr=top["std"])
    ax.set_xlabel("permutation importance (drop in Spearman)")
    ax.set_title(f"{kind.capitalize()} recommender — top 15 features")
    fig.tight_layout()
    fig.savefig(FIGURES / f"{kind}_recommender_feature_importance.png", dpi=200, bbox_inches="tight")

## Predicted vs. baseline → thesis/figures

Where each trained model and its deterministic formula agree vs. diverge,
over the full synthetic frame (not just one held-out fold, purely for the
visual). Loads the shipped models rather than refitting, so the charts match
what's actually deployed.

In [ ]:
import numpy as np

import scripts.train_recommenders as train
from src.diet import recommender

shipped = {"food": recommender.load_model(), "recipe": recommender.load_recipe_model()}
for kind in KINDS:
    model = shipped[kind]
    assert model is not None, "run scripts/train_recommenders.py first"
    rng = np.random.default_rng(train.RANDOM_STATE)
    X, y, groups, baseline, _ = train.build_training_frame(train.SECTIONS[kind], rng)
    predicted = model.predict(X)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(baseline, predicted, s=4, alpha=0.15)
    lims = [min(baseline.min(), predicted.min()), max(baseline.max(), predicted.max())]
    ax.plot(lims, lims, color="black", linewidth=1, linestyle="--", label="y = x")
    ax.set_xlabel("deterministic formula score (baseline)")
    ax.set_ylabel("trained model score")
    ax.set_title(f"{kind.capitalize()} model vs. deterministic baseline")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / f"{kind}_recommender_predicted_vs_baseline.png", dpi=200, bbox_inches="tight")